In [ ]:
using Plots
using Distributed
using LinearAlgebra

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end

@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
    using BlockDiagonals
end
using JLD2

In [3]:
type_lattice = "surface_square"

# dmin, dmax = 3, 39
# σrange = vcat(0.50:0.01:0.59, 0.591:0.001:0.607, exp(-1/2))

dmin, dmax = 3, 3
σrange = [exp(-1/2)]


drange = dmin : 2 : dmax
σdrange = []
for σ in σrange
    for d in drange
        push!(σdrange, [σ, d])
    end
end

num_samples = Int(1e4)

num_samples_each_core = Int(ceil(num_samples/num_cores))
num_total_samples = Int(num_samples_each_core * num_cores);
println([num_samples_each_core, num_samples, num_total_samples])

logfile = "$(type_lattice)_bsv_$(drange[1])_$(drange[end])_$(σrange[1])_$(σrange[end])_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

In [ ]:
@time results = pmap(1:num_cores) do _
    p_list = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])
    t_list = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])
    
    Ms = Dict()
    Ωs = Dict()
    Mperps = Dict()
    invtransposeMqs = Dict()
    invtransposeMperps = Dict()
    transposeΩMperps = Dict()
    for (ind_σd, σd) in enumerate(σdrange)
        σ, d = σd[1], Int(σd[2])
        M = surface_code_M(d) ; 
        Mperp = GKP_logical_operator_generator(M) 
        Ω = Ω_matrix(M)         
        invtransposeMq = inv(transpose(M))[1:2:end, 1:2:end]    
        Ms[d] = M
        Mperps[d] = Mperp
        Ωs[d] = Ω
        invtransposeMqs[d] = invtransposeMq
        invtransposeMperps[d] = inv(√(2π) * transpose(Mperp))
        transposeΩMperps[d] = -transpose(Ω*Mperp)
    end        
    
    for (ind_σd, σd) in enumerate(σdrange)
        σ, d = σd[1], Int(σd[2])
        p_I_bsv = 0
        time_bsv = 0        

        σdtime = @elapsed for _ in 1 : num_samples_each_core
            ξ = σ * randn(2d^2)

            if d > 21
                setprecision(BigFloat, 64)
                ξ = BigFloat.(ξ)
            end
            
            ξ2 = -√(2π) * Ms[d] * Ωs[d] * ξ
            s = ξ2 - floor.(ξ2/(2π)) * 2π
            # ηs = -transpose(Ωs[d]*Mperps[d]) * s/√(2π) ; 
            # b = inv(√(2π) * transpose(Mperps[d])) * (ηs-ξ)
            ηs = transposeΩMperps[d] * s/√(2π) ; 
            b = invtransposeMperps[d] * (ηs-ξ)
            @assert norm(round.(Int, b) - b) < 1e-10    
            
            time_bsv += @elapsed rec_q = bsv_surface_code(ηs[1:2:end], σ; Nv=5, subspace="x")

            neterror_q = invtransposeMqs[d] * (rec_q+ξ[1:2:end]) / √(2π)                

            norm(round.(Int, neterror_q) - neterror_q) < 1e-10 ? nx = 0 : nx = 1

            if mod(nx, 2) == 0
                p_I_bsv += 1
            end
        end
        p_list[[σ, d]] += p_I_bsv
        t_list[[σ, d]] += time_bsv

        if myid() == 2 # Print the progress of the 2nd worker
            println(["$(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))"])
            # open(logfile, "a") do file
            #     write(file, "$(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))\n")
            # end
        end
    end
    
    return p_list, t_list
end ;     

t_list = merge(+, [res[2] for res in results]...) 
p_list = merge(+, [res[1] for res in results]...)


map!(v->v./num_total_samples, values(t_list))
map!(v->v./num_total_samples, values(p_list))

c_list = Dict()

for (k, v) in p_list
    px = v * (1-v)
    pz = px
    py = (1-v)^2
    c_list[k] = coherent_information_pauli_channel(px, py, pz)
end

# Save the result
fn = "$(type_lattice)_bsv_$(drange[1])_$(drange[end])_$(σrange[1])_$(σrange[end])_$(num_total_samples).jld2";
jldsave(fn; 
    σrange=σrange, 
    num_samples=num_samples_each_core*num_cores,
    p_list = p_list,
    t_list = t_list,
    c_list = c_list,
    drange = drange
)

In [ ]:
sort(load(fn)["p_list"])